In [16]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import pickle

In [17]:
df = pd.read_csv("data.csv")

print("Shape:", df.shape)
df.head()

Shape: (246945, 378)


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [18]:
# Separate features
X = df.drop("diseases", axis=1)

# Convert everything to numeric (force errors → NaN)
X = X.apply(pd.to_numeric, errors='coerce')

# Replace NaN with 0
X = X.fillna(0)

# Force binary (0/1)
X = X.clip(0,1)

# Check again
print("Remaining NaN:", X.isna().sum().sum())

Remaining NaN: 0


In [19]:
y_disease = df["diseases"]

In [20]:
severity_map = {
    "panic disorder": 1,
    "depression": 1,
    "insomnia": 0,
    "shortness of breath": 2,
    "chest pain": 2
}

df["severity"] = df["diseases"].map(severity_map)

# Fill unknown diseases as medium severity
df["severity"] = df["severity"].fillna(1)

y_severity = df["severity"]

In [23]:
X_train, X_test, y_train_d, y_test_d = train_test_split(
    X, y_disease, test_size=0.2, random_state=42
)

_, _, y_train_s, y_test_s = train_test_split(
    X, y_severity, test_size=0.2, random_state=42
)

MemoryError: Unable to allocate 568. MiB for an array with shape (377, 197556) and data type int64

In [ ]:
disease_model = RandomForestClassifier(n_estimators=100)
severity_model = RandomForestClassifier(n_estimators=100)

disease_model.fit(X_train, y_train_d)
severity_model.fit(X_train, y_train_s)

In [ ]:
pred_d = disease_model.predict(X_test)
pred_s = severity_model.predict(X_test)

print("Disease Accuracy:", accuracy_score(y_test_d, pred_d))
print("Severity Accuracy:", accuracy_score(y_test_s, pred_s))

In [ ]:
pickle.dump(disease_model, open("disease.pkl","wb"))
pickle.dump(severity_model, open("severity.pkl","wb"))
pickle.dump(X.columns.tolist(), open("columns.pkl","wb"))

print("Models saved successfully")